## Imports

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage
from typing import TypedDict, Sequence, Annotated
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

## Agent State

In [23]:
class AgentState(TypedDict):
    statement: Annotated[str, "The statement on which the model generates pros and cons"]
    pros: list[str]
    cons: list[str]
    all_statements: Annotated[Sequence[str], add_messages]

## Output format for LLM

In [18]:
class LLMOutput(BaseModel):
    pros: list[str] = Field(description = 'Represents the list of positive feedbacks or pros for the given statement')
    cons: list[str] = Field(description = 'Represents the list of negative feedbacks or cons for the given statement')

llm = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

## Different ways to generate structured output from LLM

In [ ]:
# Ways to generated structured output

# 1. With default feature (not available in every model)
# llm_with_structured_output = llm.with_structured_output(LLMOutput)
# llm_with_structured_output.invoke('iphone 17 pro is launched!')

# 2. With partials and JsonOutputParser
json_parser = JsonOutputParser(pydantic_object=LLMOutput)
messages = [
    ('system', 'Generate pros and cons for the given statement by the user. Ensure {format}'),
    ('user', 'Statement: {statement}')
]
prompt = ChatPromptTemplate.from_messages(messages=messages).partial(format=json_parser.get_format_instructions())

statement = 'iphone 17 pro is launched!'

chain = prompt | llm | json_parser

# chain.invoke({'statement' : statement})


{'pros': ['Cutting-edge performance with the latest A-series processor for faster multitasking and gaming.',
  'Significant camera hardware and software improvements for professional-grade photography.',
  'Integration of advanced AI features and enhanced Apple Intelligence capabilities.',
  'Improved battery efficiency and potentially faster charging speeds.',
  'High-quality build materials and long-term software update support.'],
 'cons': ['Likely to feature a premium price tag, making it expensive for many consumers.',
  'Incremental upgrades may not justify the cost for users of recent models.',
  'Environmental impact associated with the production and disposal of electronic devices.',
  'High cost of official repairs and replacement parts.',
  'The potential removal of more ports or physical buttons may be polarizing for users.']}

## Building the graph

### 1. Create the nodes

In [20]:
def llm_node(state: AgentState) -> AgentState:
    res = chain.invoke({'statement': state['statement']})

    return {'pros': res['pros'], 'cons': res['cons'], 'all_statements': state['statement']}

### 2. Define the graph and add checkpointer

In [36]:
graph = StateGraph(AgentState)

graph.add_node('llm_node', llm_node)

graph.add_edge(START, 'llm_node')
graph.add_edge('llm_node', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

## Interact with the graph with a `thread_id`
- The `thread_id` is used to track the state of the conversation.
- Think it as a unique identifier for a specific conversation or interaction with the agent.
- By providing the same `thread_id` in subsequent interactions, you can maintain the context and state of the conversation, allowing the agent to respond appropriately based on the previous interactions.

In [ ]:
config = {
    'configurable': {
        'thread_id': 'agent_thread_1'
    }
}

while True:
    user_input = input('User: ')

    if not user_input or user_input in ('bye', 'exit', 'stop'):
        print('AI: Bye!')
        break

    print("=" * 50)
    print(f'User: {user_input}')
    print("=" * 50)
    output = workflow.invoke({'statement': user_input}, config=config)
    print(f'AI:\nPros: {output["pros"]},\nCons: {output["cons"]}')
    print("=" * 50)
    print()


print("State History:")
print(output['all_statements']) # Contains all the statements that were processed in the workflow, which can be used to track the conversation history or for debugging purposes. Using the checkpointer, we can also retrieve the state history.

User: Anthropic lanuched Opus 4.6
AI:
Pros: ['Significant improvements in reasoning and complex problem-solving capabilities.', 'Higher accuracy and reduced frequency of hallucinations in factual responses.', 'Enhanced multi-modal processing for better image and data visualization analysis.', 'Expanded context window allowing for the processing of even larger datasets and documents.', 'Better integration and developer tools for building sophisticated AI agents.'],
Cons: ['Increased cost per token making it more expensive for large-scale deployments.', 'Higher computational requirements potentially leading to increased latency compared to smaller models.', 'Potential for new safety or bias risks that require rigorous testing.', 'The need to re-evaluate and optimize existing prompts designed for older model versions.', 'Environmental concerns related to the energy consumption of training and running a more massive model.']

User: gemini launched Gemini 3.1 pro
AI:
Pros: ['Enhanced reason

## Retrieve and print the state history using the checkpointer

In [ ]:
workflow.get_state(config=config)  # Final state of the workflow execution, which includes the last processed statement, pros, and cons.

StateSnapshot(values={'statement': 'gemini launched Gemini 3.1 pro', 'pros': ['Enhanced reasoning and problem-solving capabilities across complex tasks.', 'Significant expansion of the context window, allowing for processing of massive datasets or long documents.', 'Improved multimodal performance for better understanding of text, images, video, and audio.', 'Higher efficiency and faster response times for enterprise-level applications.', 'Seamless integration into the existing Google ecosystem and cloud productivity tools.'], 'cons': ['Potential for increased subscription or API usage costs for accessing premium features.', 'Ongoing risks of AI hallucinations and the generation of factually incorrect information.', 'High computational resource requirements for large-scale deployment and integration.', 'Privacy and data security concerns regarding the handling of sensitive user data for model refinement.', 'Possible steep learning curve for developers to optimize applications for the n

In [ ]:
# Retrieve and print the state history using the checkpointer
for idx, state in enumerate(workflow.get_state_history(config=config)):
    print(f"Step {idx + 1}:")
    print(state)
    print("-" * 50)

Step 1:
StateSnapshot(values={'statement': 'gemini launched Gemini 3.1 pro', 'pros': ['Enhanced reasoning and problem-solving capabilities across complex tasks.', 'Significant expansion of the context window, allowing for processing of massive datasets or long documents.', 'Improved multimodal performance for better understanding of text, images, video, and audio.', 'Higher efficiency and faster response times for enterprise-level applications.', 'Seamless integration into the existing Google ecosystem and cloud productivity tools.'], 'cons': ['Potential for increased subscription or API usage costs for accessing premium features.', 'Ongoing risks of AI hallucinations and the generation of factually incorrect information.', 'High computational resource requirements for large-scale deployment and integration.', 'Privacy and data security concerns regarding the handling of sensitive user data for model refinement.', 'Possible steep learning curve for developers to optimize applications f

In [ ]:
# Timetravel, we can resume the workflow from any previous state using the checkpoint_id and config with the same thread_id. It is useful when the workflow execution is interrupted due to any reason and we want to resume it from the last state instead of starting from the beginning.

workflow.invoke(None, config={'checkpoint_id': '1f1397f5-a2b4-69b0-8000-fb6d82b2e95c', 'configurable': {'thread_id': 'agent_thread_1'}})

{'statement': 'Anthropic lanuched Opus 4.6',
 'pros': ['Significant advancement in reasoning and cognitive processing capabilities',
  'Improved performance in complex coding, mathematics, and creative writing tasks',
  'Enhanced context window capacity for processing extremely large datasets or documents',
  'Higher levels of accuracy with a reduced likelihood of factual hallucinations',
  'Better steerability and adherence to complex multi-step instructions'],
 'cons': ['Potentially higher cost per million tokens compared to previous models',
  'Possible increase in latency or response times due to model size and complexity',
  'Increased computational resource requirements for API calls or local hosting',
  'Existing prompts and workflows may need optimization to leverage the new architecture',
  'Ongoing risks associated with model alignment and potential biases in larger datasets'],
 'all_statements': [HumanMessage(content='Anthropic lanuched Opus 4.6', additional_kwargs={}, respo

In [ ]:
workflow.get_state(config=config) # The final state of the workflow execution after resuming from the checkpoint

StateSnapshot(values={'statement': 'Anthropic lanuched Opus 4.6', 'pros': ['Significant advancement in reasoning and cognitive processing capabilities', 'Improved performance in complex coding, mathematics, and creative writing tasks', 'Enhanced context window capacity for processing extremely large datasets or documents', 'Higher levels of accuracy with a reduced likelihood of factual hallucinations', 'Better steerability and adherence to complex multi-step instructions'], 'cons': ['Potentially higher cost per million tokens compared to previous models', 'Possible increase in latency or response times due to model size and complexity', 'Increased computational resource requirements for API calls or local hosting', 'Existing prompts and workflows may need optimization to leverage the new architecture', 'Ongoing risks associated with model alignment and potential biases in larger datasets'], 'all_statements': [HumanMessage(content='Anthropic lanuched Opus 4.6', additional_kwargs={}, res